# Modeling and Evaluating 

Do có nhiều sự lựa chọn nên ở đây tụi em quyết định chọn LightGBM do có hiệu năng và cũng là model mạnh nhất hiện tại

Sau nhiều lần thử nghiệm với các model khác thì thấy LightGBM hiệu suất hơn hẳn

- evaluating scale với wrmsse 
- tại sao lại chọn những tham số đó 
- lí do chọn lgbm - ưu chuộng và hiệu quả nhất 
- test với model khác - cân nhắc
- 

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
import os, sys, gc, time, warnings, pickle, psutil, random, optuna
from sklearn.metrics import mean_squared_error
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Details:", tf.config.experimental.list_physical_devices('GPU'))

Num GPUs Available: 1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
df = pd.read_parquet('../../data/train_val.parquet')
print(df.head())

def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    
    for i, t in enumerate(types):
        col = cols[i]
        if 'int' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif 'float' in str(t):
            c_min = df[col].min()
            c_max = df[col].max()
            # ⚠️ Không dùng float16 vì không lưu được trong Parquet
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)    
    return df

df = downcast(df)
print(df.columns)


                   id  item_id  store_id  state_id  dept_id  cat_id     d  \
0  HOBBIES_1_001_CA_1     1438         1         1        4       2  1029   
1  HOBBIES_1_002_CA_1     1439         1         1        4       2  1029   
2  HOBBIES_1_003_CA_1     1440         1         1        4       2  1029   
3  HOBBIES_1_004_CA_1     1441         1         1        4       2  1029   
4  HOBBIES_1_005_CA_1     1442         1         1        4       2  1029   

   demand  lag_t7  rolling_mean_lag7_w7  ...  event_name_2  snap_CA  snap_TX  \
0     0.0     0.0              1.000000  ...             1        0        0   
1     0.0     1.0              0.142857  ...             1        0        0   
2     0.0     0.0              0.000000  ...             1        0        0   
3     3.0     0.0              1.142857  ...             1        0        0   
4     1.0     1.0              0.428571  ...             1        0        0   

   snap_WI sell_price  disaster_type  disaster_severity_

In [3]:
null_columns = df.columns[df.isna().any()].tolist()

# Kiểm tra giá trị 0
zero_columns = df.columns[(df == 0).any()].tolist()

print("Các cột có giá trị null:", null_columns)
print("Các cột có giá trị 0:", zero_columns)

Các cột có giá trị null: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'sell_price', 'disaster_type']
Các cột có giá trị 0: ['demand', 'lag_t7', 'rolling_mean_lag7_w7', 'rolling_mean_lag7_w28', 'lag_t28', 'rolling_mean_lag28_w7', 'rolling_mean_lag28_w28', 'snap_CA', 'snap_TX', 'snap_WI', 'disaster_severity_flag', 'flag_ufc', 'gold_price']


In [4]:
# def fill_gold_price_with_rolling_mean(df):
#     df = df.copy()
    
#     for i in range(len(df)):
#         # Kiểm tra nếu giá trị là null hoặc bằng 0
#         if pd.isna(df.loc[i, 'gold_price']) or df.loc[i, 'gold_price'] == 0:
#             if i == 0:  # Trường hợp đặc biệt: hàng đầu tiên
#                 # Đối với hàng đầu tiên, có thể sử dụng giá trị mặc định hoặc để nguyên
#                 continue  # Bỏ qua và giữ nguyên giá trị
#             else:
#                 # Lấy tối đa 7 ngày trước để tính trung bình
#                 start_idx = max(0, i-7)
#                 # Lọc những giá trị hợp lệ (không null và khác 0)
#                 valid_values = [df.loc[j, 'gold_price'] for j in range(start_idx, i) 
#                                 if not pd.isna(df.loc[j, 'gold_price']) and df.loc[j, 'gold_price'] != 0]
                
#                 if valid_values:  # Nếu có ít nhất một giá trị hợp lệ
#                     # Tính trung bình và gán
#                     df.loc[i, 'gold_price'] = sum(valid_values) / len(valid_values)
#                 elif i > 0:  # Nếu không có giá trị hợp lệ nhưng có hàng trước đó
#                     # Sử dụng giá trị của hàng liền trước (nếu có)
#                     df.loc[i, 'gold_price'] = df.loc[i-1, 'gold_price']
    
#     return df

# # Gọi hàm
# df = fill_gold_price_with_rolling_mean(df)

In [5]:
# df.to_parquet('../../data/train_val.parquet')

In [6]:
df = df.drop(['date','disaster_type'],axis=1)
sample_submission = pd.read_csv('../../data/sample_submission.csv')
sample_submission = downcast(sample_submission)

LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942 # first to predict
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")
    
    return out.reset_index()

def prep_train(df,drop_d = 1000):

    x = list(set(df.columns) - {'id', 'd', 'demand'}) # exclude 'date' from features
    print(x)
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]

    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    train = lgb.Dataset(xtrain, label = ytrain)
    valid = lgb.Dataset(xvalid, label = yvalid)

    # return validation data for Optuna
    return train, valid, test, x, xvalid, yvalid



def pred_all(fit, test, x):
    """ Calculate predictions """
    
    # Recursive prediction
    for i, day in enumerate(np.arange(FIRST, FIRST + LENGTH)):
        test_day = demand_features_eval(test[(test.d <= day) & (test.d >= day - max(LAGS) - max(WINDOWS))])
        test.loc[test.d == day, "demand"] = fit.predict(test_day[x])
    
    return test

def pred_to_csv(test, cols=sample_submission.columns, file="submission.csv"):
    """ Reshape predictions and save submission csv """
     
    # Prepare for reshaping
    test = test.assign(id=test.id + "_" + np.where(test.d < FIRST, "validation", "evaluation"),
                       F="F" + (test.d - FIRST + LENGTH + 1 - LENGTH * (test.d >= FIRST)).astype("str"))
    
    # Reshape
    submission = test.pivot(index="id", columns="F", values="demand").reset_index()[cols].fillna(1)
    
    # Export
    submission.to_csv(file, index=False)
    
    return True

train, valid, test, x, X_valid, y_valid = prep_train(df, 1000 - 28)

del df
gc.collect()

['snap_WI', 'store_id', 'rolling_mean_lag28_w28', 'month', 'disaster_severity_flag', 'flag_ufc', 'year', 'rolling_mean_lag7_w28', 'wday', 'snap_TX', 'dept_id', 'sell_price', 'rolling_mean_lag7_w7', 'lag_t28', 'event_name_2', 'humidity', 'snap_CA', 'cat_id', 'state_id', 'rolling_mean_lag28_w7', 'item_id', 'lag_t7', 'gold_price', 'event_name_1']


0

In [ ]:
def objective(trial):
    # Gợi ý các hyperparameter
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'feature_pre_filter': False,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 1000, 3000),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    # Train
    gbm = lgb.train(
        param,
        train,
        valid_sets=[valid],
        num_boost_round=100,
    )

    preds = gbm.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))

    return rmse  # Tối ưu RMSE → minimize

# Study
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30)

# Kết quả
print("Best trial:")
trial = study.best_trial
print(f"  RMSE: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-05-05 11:56:43,791] A new study created in memory with name: no-name-e7a0c3a7-e365-4980-88a6-eee9fedbfe3c


[1]	valid_0's rmse: 3.66012
[2]	valid_0's rmse: 3.65436
[3]	valid_0's rmse: 3.64873
[4]	valid_0's rmse: 3.64306
[5]	valid_0's rmse: 3.63737
[6]	valid_0's rmse: 3.63178
[7]	valid_0's rmse: 3.62619
[8]	valid_0's rmse: 3.62057
[9]	valid_0's rmse: 3.61499
[10]	valid_0's rmse: 3.60962
[11]	valid_0's rmse: 3.60408
[12]	valid_0's rmse: 3.59857
[13]	valid_0's rmse: 3.59306
[14]	valid_0's rmse: 3.58762
[15]	valid_0's rmse: 3.58362
[16]	valid_0's rmse: 3.57866
[17]	valid_0's rmse: 3.57326
[18]	valid_0's rmse: 3.56793
[19]	valid_0's rmse: 3.56256
[20]	valid_0's rmse: 3.55722
[21]	valid_0's rmse: 3.55192
[22]	valid_0's rmse: 3.54657
[23]	valid_0's rmse: 3.54119
[24]	valid_0's rmse: 3.53592
[25]	valid_0's rmse: 3.5306
[26]	valid_0's rmse: 3.52579
[27]	valid_0's rmse: 3.52062
[28]	valid_0's rmse: 3.51536
[29]	valid_0's rmse: 3.51023
[30]	valid_0's rmse: 3.50507
[31]	valid_0's rmse: 3.49989
[32]	valid_0's rmse: 3.49548
[33]	valid_0's rmse: 3.49044
[34]	valid_0's rmse: 3.48542
[35]	valid_0's rmse: 3.4